In [0]:
from pyspark.sql import functions as F

BRONZE           = "immo.bronze.dvf_mutations"
SILVER_LIGNES    = "immo.silver.dvf_lignes"
SILVER_MUTATIONS = "immo.silver.dvf_mutations"

# --- Helpers de conversion ---------------------------------------------------
# Le serverless tourne en mode ANSI : un cast invalide fait planter le job.
# On utilise donc try_cast / try_to_timestamp, qui renvoient NULL au lieu d'échouer.
def to_decimal(col, precision="15,2"):
    return F.expr(f"try_cast(regexp_replace({col}, ',', '.') AS DECIMAL({precision}))")

def to_int(col):
    return F.expr(f"try_cast({col} AS INT)")

# --- 1. Lecture + dédoublonnage des lignes strictement identiques ------------
bronze = spark.table(BRONZE)
colonnes_metier = [c for c in bronze.columns if not c.startswith("_")]
bronze_dedup = bronze.dropDuplicates(colonnes_metier)

# --- 2. Codes géographiques -------------------------------------------------
dep = F.col("code_departement")
code_insee = (
    F.when(F.length(dep) == 3,   # DOM (971, 972...) : 97 + 1 + 2 derniers chiffres commune
           F.concat(dep, F.substring(F.lpad("code_commune", 3, "0"), 2, 2)))
     .otherwise(F.concat(F.lpad(dep, 2, "0"), F.lpad("code_commune", 3, "0")))
)

# Identifiant cadastral officiel sur 14 caractères : INSEE + préfixe + section + n° de plan
id_parcelle = F.concat(
    code_insee,
    F.lpad(F.coalesce(F.col("prefixe_de_section"), F.lit("000")), 3, "0"),
    F.lpad("section", 2, "0"),
    F.lpad("no_plan", 4, "0"),
)

# --- 3. Typage et normalisation ---------------------------------------------
lignes = bronze_dedup.select(
    F.expr("to_date(try_to_timestamp(date_mutation, 'dd/MM/yyyy'))").alias("date_mutation"),
    F.col("nature_mutation"),
    to_decimal("valeur_fonciere").alias("valeur_fonciere"),
    F.col("no_disposition"),
    F.lpad(dep, 2, "0").alias("code_departement"),
    code_insee.alias("code_insee"),
    F.col("commune"),
    F.lpad("code_postal", 5, "0").alias("code_postal"),
    F.concat_ws(" ", "no_voie", "b_t_q", "type_de_voie", "voie").alias("adresse"),
    id_parcelle.alias("id_parcelle"),
    to_int("code_type_local").alias("code_type_local"),
    F.col("type_local"),
    to_int("surface_reelle_bati").alias("surface_reelle_bati"),
    to_int("nombre_pieces_principales").alias("nombre_pieces"),
    to_decimal("surface_carrez_du_1er_lot", "10,2").alias("surface_carrez_lot1"),
    to_int("nombre_de_lots").alias("nombre_de_lots"),
    F.col("nature_culture"),
    to_int("surface_terrain").alias("surface_terrain"),
    F.col("_source_file"),
)

# --- 4. Reconstruction de l'identifiant de vente ----------------------------
lignes = lignes.withColumn(
    "mutation_id",
    F.sha2(F.concat_ws("|", "date_mutation", "nature_mutation", "valeur_fonciere",
                       "no_disposition", "code_insee"), 256),
)

(lignes.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(SILVER_LIGNES))

print(f"{spark.table(SILVER_LIGNES).count():,} lignes écrites dans {SILVER_LIGNES}")

In [0]:
lignes = spark.table(SILVER_LIGNES)
tl = F.col("type_local")

def compte(type_):
    return F.sum(F.when(tl == type_, 1).otherwise(0))

# Attributs de la vente et comptage des locaux
ventes = lignes.groupBy("mutation_id").agg(
    F.first("date_mutation").alias("date_mutation"),
    F.first("nature_mutation").alias("nature_mutation"),
    F.first("valeur_fonciere").alias("valeur_fonciere"),
    F.first("code_departement").alias("code_departement"),
    F.first("code_insee").alias("code_insee"),
    F.first("commune").alias("commune"),
    F.first("code_postal", ignorenulls=True).alias("code_postal"),
    F.countDistinct("id_parcelle").alias("nb_parcelles"),
    compte("Maison").alias("nb_maisons"),
    compte("Appartement").alias("nb_appartements"),
    compte("Dépendance").alias("nb_dependances"),
    F.sum(F.when(tl.startswith("Local"), 1).otherwise(0)).alias("nb_locaux_activite"),
    F.sum(F.when(tl.isin("Maison", "Appartement"), F.col("surface_reelle_bati"))).alias("surface_habitable"),
    F.sum(F.when(tl.isin("Maison", "Appartement"), F.col("nombre_pieces"))).alias("nb_pieces"),
)

# Surface de terrain : une parcelle est répétée pour chaque local qu'elle porte,
# donc on la compte une seule fois par (parcelle, nature de culture)
terrains = (
    lignes.where(F.col("surface_terrain").isNotNull())
          .dropDuplicates(["mutation_id", "id_parcelle", "nature_culture", "surface_terrain"])
          .groupBy("mutation_id")
          .agg(F.sum("surface_terrain").alias("surface_terrain"))
)

mutations = (
    ventes.join(terrains, "mutation_id", "left")
          .withColumn("type_bien",
              F.when((F.col("nb_maisons") == 1) & (F.col("nb_appartements") == 0) & (F.col("nb_locaux_activite") == 0), "Maison")
               .when((F.col("nb_appartements") == 1) & (F.col("nb_maisons") == 0) & (F.col("nb_locaux_activite") == 0), "Appartement")
               .when(F.col("nb_maisons") + F.col("nb_appartements") + F.col("nb_locaux_activite") == 0, "Sans bâti")
               .otherwise("Multiple / mixte"))
)

(mutations.write.mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(SILVER_MUTATIONS))

print(f"{spark.table(SILVER_MUTATIONS).count():,} ventes dans {SILVER_MUTATIONS}")

In [0]:
%sql
-- Combien de lignes le nettoyage a-t-il fait disparaître ?
SELECT
  (SELECT COUNT(*) FROM immo.bronze.dvf_mutations) AS lignes_bronze,
  (SELECT COUNT(*) FROM immo.silver.dvf_lignes)    AS lignes_silver,
  (SELECT COUNT(*) FROM immo.silver.dvf_mutations) AS ventes;

-- Répartition des ventes par type de bien
SELECT type_bien, COUNT(*) AS nb, ROUND(PERCENTILE(valeur_fonciere, 0.5)) AS prix_median
FROM immo.silver.dvf_mutations
WHERE nature_mutation = 'Vente'
GROUP BY type_bien ORDER BY nb DESC;

-- Taux de valeurs manquantes sur les champs clés
SELECT
  ROUND(AVG(CASE WHEN date_mutation   IS NULL THEN 1 ELSE 0 END) * 100, 2) AS pct_date_null,
  ROUND(AVG(CASE WHEN valeur_fonciere IS NULL THEN 1 ELSE 0 END) * 100, 2) AS pct_prix_null,
  ROUND(AVG(CASE WHEN code_postal     IS NULL THEN 1 ELSE 0 END) * 100, 2) AS pct_cp_null
FROM immo.silver.dvf_mutations;

In [0]:
%sql
SELECT nature_mutation,
       COUNT(*) AS nb_ventes,
       SUM(CASE WHEN valeur_fonciere IS NULL THEN 1 ELSE 0 END) AS prix_null,
       SUM(CASE WHEN code_postal     IS NULL THEN 1 ELSE 0 END) AS cp_null,
       SUM(CASE WHEN code_postal IS NULL AND type_bien = 'Sans bâti' THEN 1 ELSE 0 END) AS cp_null_sans_bati
FROM immo.silver.dvf_mutations
GROUP BY nature_mutation
ORDER BY nb_ventes DESC;